In [4]:
import pandas as pd
pd.set_option("display.max_columns", None)   # 显示所有列
pd.set_option("display.max_rows", 200)       # 可选：显示更多行
pd.set_option("display.width", None)         # 自动适配宽度
pd.set_option("display.max_colwidth", None)  # 不截断单元格内容

In [2]:
import pandas as pd
pd.set_option("display.max_columns", None)   # 显示所有列

In [3]:
import gzip
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
from Bio.Seq import Seq

# =========================
# 1. 路径设置
# =========================

orf_fna = "/disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/Ribo-seq_ORFs.primary.fna"
orf_faa = "/disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/Ribo-seq_ORFs.primary.faa"   # 如果没有，可以设为 None
tx_fa   = "/disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/gencode.v45.transcripts.fa"

out_csv = "/disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/gencode_phase2_primary_flank450_for_esm2.csv.gz"

FLANK = 450
LABEL_RAW = "GENCODE_Phase2_Primary"
OUTPUT_RNA = False   # 如果后面给 RNA-FM 用，可以改成 True，把 T 转成 U


# =========================
# 2. 小工具函数
# =========================

def open_maybe_gz(path, mode="rt"):
    if path.endswith(".gz"):
        return gzip.open(path, mode)
    return open(path, mode)


def clean_seq(seq):
    """统一成大写 DNA 格式"""
    return str(seq).upper().replace("U", "T")


def to_output_seq(seq):
    """是否输出成 RNA 序列"""
    seq = clean_seq(seq)
    return seq.replace("T", "U") if OUTPUT_RNA else seq


def parse_orf_header(rec):
    """
    ORF header:
    >c1riboseqorf662:LINC00115:lncRNA gencode=v45 assembly=GRCh38
    """
    main = rec.id.split()[0]
    parts = main.split(":")

    return {
        "orf_id": parts[0],
        "gene_symbol": parts[1] if len(parts) > 1 else None,
        "orf_type": parts[2] if len(parts) > 2 else None,
        "orf_name": main,
    }


def parse_tx_header(rec):
    """
    transcript header:
    >ENST00000492842.2|ENSG00000240361.3|OTTHUMG...|OTTHUMT...|OR4G11P-201|OR4G11P|939|transcribed_unprocessed_pseudogene|
    """
    parts = rec.id.split("|")

    tx_id_version = parts[0]
    gene_id_version = parts[1] if len(parts) > 1 else None
    transcript_name = parts[4] if len(parts) > 4 else None
    gene_name = parts[5] if len(parts) > 5 else None
    tx_len = int(parts[6]) if len(parts) > 6 and parts[6].isdigit() else len(rec.seq)
    transcript_type = parts[7] if len(parts) > 7 else None

    return {
        "transcript_id_version": tx_id_version,
        "transcript_id": tx_id_version.split(".")[0],
        "gene_id_version": gene_id_version,
        "gene_id": gene_id_version.split(".")[0] if gene_id_version else None,
        "transcript_name": transcript_name,
        "gene_name": gene_name,
        "tx_len": tx_len,
        "transcript_type": transcript_type,
    }


def translate_orf(seq):
    """备用：如果没有 faa 文件，就从 fna 翻译蛋白序列"""
    seq = clean_seq(seq)
    aa = str(Seq(seq).translate(to_stop=False))
    return aa.rstrip("*")


# =========================
# 3. 读取 ORF 蛋白序列 faa
# =========================

aa_dict = {}

if orf_faa is not None:
    with open_maybe_gz(orf_faa) as f:
        for rec in SeqIO.parse(f, "fasta"):
            orf_id = rec.id.split()[0].split(":")[0]
            aa_dict[orf_id] = str(rec.seq).replace("*", "")

print("loaded aa seqs:", len(aa_dict))


# =========================
# 4. 读取 transcript.fa，按 gene_name 建索引
# =========================

tx_by_gene = defaultdict(list)

with open_maybe_gz(tx_fa) as f:
    for rec in SeqIO.parse(f, "fasta"):
        meta = parse_tx_header(rec)
        seq = clean_seq(rec.seq)

        item = {
            **meta,
            "tx_seq": seq,
        }

        # 主要用 gene_name 建索引
        if meta["gene_name"]:
            tx_by_gene[meta["gene_name"]].append(item)

        # 兼容部分 ORF header 里 gene_symbol 是 ENSG 的情况
        if meta["gene_id"]:
            tx_by_gene[meta["gene_id"]].append(item)

print("indexed genes:", len(tx_by_gene))


# =========================
# 5. 读取 ORF.fna，在同 gene transcripts 中匹配
# =========================

rows = []

with open_maybe_gz(orf_fna) as f:
    for rec in SeqIO.parse(f, "fasta"):
        orf_meta = parse_orf_header(rec)
        orf_seq = clean_seq(rec.seq)
        gene = orf_meta["gene_symbol"]

        candidates = tx_by_gene.get(gene, [])

        best_hit = None

        # 优先检查最长 transcript
        candidates = sorted(candidates, key=lambda x: x["tx_len"], reverse=True)

        for tx in candidates:
            tx_seq = tx["tx_seq"]
            start = tx_seq.find(orf_seq)

            if start == -1:
                continue

            end = start + len(orf_seq)

            upstream = tx_seq[max(0, start - FLANK):start]
            downstream = tx_seq[end:end + FLANK]

            best_hit = {
                **orf_meta,
                "sample_id": orf_meta["orf_id"],
                "source": "GENCODE_Phase2",
                "label_raw": LABEL_RAW,
                "binary_label": 1,

                "transcript_id": tx["transcript_id"],
                "transcript_id_version": tx["transcript_id_version"],
                "gene_id": tx["gene_id"],
                "gene_id_version": tx["gene_id_version"],
                "transcript_name": tx["transcript_name"],
                "transcript_type": tx["transcript_type"],
                "tx_len": tx["tx_len"],

                "orf_start_tx_0based": start,
                "orf_end_tx_0based": end,

                "upstream_seq_450nt": to_output_seq(upstream),
                "orf_seq": to_output_seq(orf_seq),
                "downstream_seq_450nt": to_output_seq(downstream),

                "upstream_len": len(upstream),
                "orf_nt_len": len(orf_seq),
                "downstream_len": len(downstream),

                "aa_seq": aa_dict.get(orf_meta["orf_id"], translate_orf(orf_seq)),
                "protein_sequence": aa_dict.get(orf_meta["orf_id"], translate_orf(orf_seq)),
                "protein length (a.a.)": len(aa_dict.get(orf_meta["orf_id"], translate_orf(orf_seq))),

                "n_candidate_transcripts": len(candidates),
                "matched": True,
            }

            # 因为 candidates 已经按 tx_len 从长到短排序，
            # 找到第一个能匹配的就是最长可匹配 transcript
            break

        if best_hit is None:
            best_hit = {
                **orf_meta,
                "sample_id": orf_meta["orf_id"],
                "source": "GENCODE_Phase2",
                "label_raw": LABEL_RAW,
                "binary_label": 1,

                "transcript_id": None,
                "transcript_id_version": None,
                "gene_id": None,
                "gene_id_version": None,
                "transcript_name": None,
                "transcript_type": None,
                "tx_len": None,

                "orf_start_tx_0based": None,
                "orf_end_tx_0based": None,

                "upstream_seq_450nt": None,
                "orf_seq": to_output_seq(orf_seq),
                "downstream_seq_450nt": None,

                "upstream_len": 0,
                "orf_nt_len": len(orf_seq),
                "downstream_len": 0,

                "aa_seq": aa_dict.get(orf_meta["orf_id"], translate_orf(orf_seq)),
                "protein_sequence": aa_dict.get(orf_meta["orf_id"], translate_orf(orf_seq)),
                "protein length (a.a.)": len(aa_dict.get(orf_meta["orf_id"], translate_orf(orf_seq))),

                "n_candidate_transcripts": len(candidates),
                "matched": False,
            }

        rows.append(best_hit)


# =========================
# 6. 保存结果
# =========================

df = pd.DataFrame(rows)

df.to_csv(out_csv, index=False)

print("saved:", out_csv)
print("shape:", df.shape)
print(df["matched"].value_counts(dropna=False))
print(df["orf_type"].value_counts(dropna=False).head(20))

print(
    df[
        [
            "sample_id",
            "gene_symbol",
            "orf_type",
            "transcript_id",
            "transcript_type",
            "upstream_len",
            "orf_nt_len",
            "downstream_len",
            "protein length (a.a.)",
            "matched",
        ]
    ].head()
)

loaded aa seqs: 10127
indexed genes: 104200
saved: /disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/gencode_phase2_primary_flank450_for_esm2.csv.gz
shape: (10127, 28)
matched
True    10127
Name: count, dtype: int64
orf_type
uORF      6437
lncRNA    2108
dORF       657
PT         538
uoORF      267
intORF     107
doORF       13
Name: count, dtype: int64
         sample_id gene_symbol orf_type    transcript_id transcript_type  \
0  c1riboseqorf662   LINC00115   lncRNA  ENST00000473798          lncRNA   
1  c1riboseqorf663   LINC01128   lncRNA  ENST00000670780          lncRNA   
2  c1riboseqorf667   LINC01128   lncRNA  ENST00000670780          lncRNA   
3  c1riboseqorf668   LINC01128   lncRNA  ENST00000670780          lncRNA   
4  c1riboseqorf670   LINC01128   lncRNA  ENST00000670780          lncRNA   

   upstream_len  orf_nt_len  downstream_len  protein length (a.a.)  matched  
0           450          27             450                      8     True  
1      

In [10]:
import pandas as pd

out_csv = "/disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/gencode_phase2_primary_flank450_for_esm2.csv.gz"

table_gencode_phase2 = pd.read_csv(out_csv)

# 1. 蛋白长度转成数值
table_gencode_phase2["protein length (a.a.)"] = pd.to_numeric(
    table_gencode_phase2["protein length (a.a.)"], errors="coerce"
)

# 2. 提取 ORF 起始密码子
table_gencode_phase2["start_codon"] = (
    table_gencode_phase2["orf_seq"]
    .astype(str)
    .str.upper()
    .str.replace("U", "T", regex=False)
    .str[:3]
)

# 3. 筛选：30-150 aa + ATG起始 + 成功匹配到转录本
table_gencode_phase2_slim_atg = table_gencode_phase2[
    (table_gencode_phase2["protein length (a.a.)"] >= 0) &
    (table_gencode_phase2["protein length (a.a.)"] <= 150) &
    (table_gencode_phase2["start_codon"] == "ATG") &
    (table_gencode_phase2["matched"] == True) &
    (table_gencode_phase2["matched"].astype(str).str.lower().eq("true"))
].copy()

#    (table_gencode_phase2["protein length (a.a.)"] >= 30) &
#    (table_gencode_phase2["protein length (a.a.)"] <= 150) &

print(table_gencode_phase2.shape)
print(table_gencode_phase2_slim.shape)

print(table_gencode_phase2_slim["start_codon"].value_counts(dropna=False))

display(table_gencode_phase2_slim.head(n=20))

def pick_col(df, cols):
    for c in cols:
        if c in df.columns:
            return c
    raise ValueError(f"找不到这些列: {cols}")

orf_col  = pick_col(table_gencode_phase2_slim_atg, ["orf_seq"])
up_col   = pick_col(table_gencode_phase2_slim_atg, ["upstream_seq_450nt", "upstream_450nt", "upstream_seq_50nt"])
down_col = pick_col(table_gencode_phase2_slim_atg, ["downstream_seq_450nt", "downstream_450nt_after_stop", "downstream_seq_50nt"])

test_gencode = pd.DataFrame({
    "sample_id": table_gencode_phase2_slim_atg["orf_id"].astype(str),
    "protein_accession": table_gencode_phase2_slim_atg["gene_symbol"].astype(str),
    "transcript_accession": table_gencode_phase2_slim_atg["transcript_id"].astype(str),
    "protein_len_aa": (table_gencode_phase2_slim_atg[orf_col].astype(str).str.len() // 3 - 1),
    "orf_seq": table_gencode_phase2_slim_atg[orf_col].astype(str),
    "upstream_seq_50nt": table_gencode_phase2_slim_atg[up_col].astype(str),
    "downstream_seq_50nt": table_gencode_phase2_slim_atg[down_col].astype(str),
    "orf_nt_len": table_gencode_phase2_slim_atg[orf_col].astype(str).str.len(),
    "label_raw": "gencode",
    "protein_len_a": (table_gencode_phase2_slim_atg[orf_col].astype(str).str.len() // 3 - 1),
    "label_id": 8,
})

print(test_gencode.shape)
display(test_gencode.head())

###输出成表格###
out_pkl = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.pkl"
out_csv = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.csv.gz"
test_gencode.to_pickle(out_pkl)
test_gencode.to_csv(out_csv, index=False, compression="gzip")

print(out_pkl)
print(out_csv)

(10127, 29)
(1194, 29)
start_codon
ATG    1194
Name: count, dtype: int64


,orf_id,gene_symbol,orf_type,orf_name,sample_id,source,label_raw,binary_label,transcript_id,transcript_id_version,gene_id,gene_id_version,transcript_name,transcript_type,tx_len,orf_start_tx_0based,orf_end_tx_0based,upstream_seq_450nt,orf_seq,downstream_seq_450nt,upstream_len,orf_nt_len,downstream_len,aa_seq,protein_sequence,protein length (a.a.),n_candidate_transcripts,matched,start_codon
7,c1riboseqorf6,UBE2J2,dORF,c1riboseqorf6:UBE2J2:dORF,c1riboseqorf6,GENCODE_Phase2,GENCODE_Phase2_Primary,1,ENST00000450390,ENST00000450390.6,ENSG00000160087,ENSG00000160087.21,UBE2J2-208,nonsense_mediated_decay,2455,1310,1409,TCCTGAAGTCGTGGAGGAGATTAAACAAAAACAGAAAGCACAAGACGAACTCAGTAGCAGACCCCAGACTCTCCCCTTGCCAGACGTGGTTCCAGACGGGGAGACGCACCTCGTCCAGAACGGGATTCAGCTGCTCAACGGGCATGCGCCGGGGGCCGTCCCAAACCTCGCAGGGCTCCAGCAGGCCAACCGGCACCACGGACTCCTGGGTGGCGCCCTGGCGAACTTGTTTGTGATAGTTGGGTTTGCAGCCTTTGCTTACACGGTCAAGTACGTGCTGAGGAGCATCGCGCAGGAGTGAGGCCCAGGCGCCGAGACCCAAGGCGCCACTGAGGGCACCGCGCACCAGAGCGTGACCTCGGCAGGCTGGACACACTGCCCAGCACAGGCAGACCCACCAGGCTCCTAGGTTTAGCTTTTAAAAACCTGAAAGGGGAAGCAAAAACCAAA,ATGTGTGACTGGGCTTTGGAGGAGACTGGAGCCTCAGCCCTGTCCTGGCCACGGGCCGCTGGGGCTGGTGTGGGTGGGCCTTGTGTGCTGGATTTGTAG,CTTATCTTCCGTGTTGTCTTTGGACCTGTTTTAGTAAACCCGTTTTTCATTTTATTAGATGTGGTCACTTAGAAATGCAAACTTGCTGCCGACCGCGGGCTGCTCCTGCGTTCTTGGAGCTCCTGGCGCGTTTCTCGGAGCTCCCGGCTCCTCAGCGGGTGGGAACCTCGGGGCCCAGGGGTGGAGCTGGCGTCCGCGGGTGCTGGTCTGGCCTGGCCGTGTGGTGATGAGGCTTAGCGGGGCCAGTGACGGCCGTGGCTCAGGATCCATAAGTCGGGGTTTGGTCTCAGCATTTACAAATGTGTTTACAGTCAGAATGAAACACATTCCTTCTAGAAAGTGCTTGGGGGTTTTTGCTGCCCTGGAAGCCAGGAGCCTGCTCACTCCAACCACAAGTCGCCCTTGACTGCGGCGGCCGCGAGCGGGGCGGGGGCTGCCGGTGCCCTCCGC,450,99,450,MCDWALEETGASALSWPRAAGAGVGGPCVLDL,MCDWALEETGASALSWPRAAGAGVGGPCVLDL,32,20,True,ATG
14,c1riboseqorf313,ENSG00000272106,lncRNA,c1riboseqorf313:ENSG00000272106:lncRNA,c1riboseqorf313,GENCODE_Phase2,GENCODE_Phase2_Primary,1,ENST00000607222,ENST00000607222.1,ENSG00000272106,ENSG00000272106.1,ENST00000607222,lncRNA,2038,334,445,GCGCAGCCCCAGCCCATCCCGTGCGGACCCTACGCCGAGTCCAGCCCGGGGCTGCTGGGCCAGCGGGAGCGGGAACCGGACGGCGCTCGCGTTCTGCCAGAGCGGAGAACGACTCGGGGACGGTGCGGAGGACCAGGGGAGGAGATCCGCGCCCGCACGCACCATCAGCCGAGAGGACGGCCCGCATCGCGCCAGTGACCTAGGAGCGACTCGGGGACGGCTTCGGGCTCCCAGGCCCTCCTGGGCCACCGGGCACTGAGGAGAGCGGCGGCTGGAAACTGTGGGACCCGAAGGGCGGGAGCCCAGGGCCCGCCCCCGCCGGAGCGCCCCCGCC,ATGGCCACGCCCCCGCCACGGGGATGGGCAGGGCAAGCACCGGAAGTGCGTCACGCGGGAGCGTCTGCGTCGCCTACCCTGCGTGGCGCAGCTAGGTTCCGAGCGGCTTAG,CCTCCGGGCGCCTGCGCAGTGGCGGTCCTAGAGACGCACTCGGTTGCCATAGAGACGGGGGCTTGGGAGCGACGTCGCGCTCGCTGGCACTGGCGGCACTTGCCGCGCCGCAGCCACCTCTTTCCGACTCCACCAGTGCGGGCCCTATCCGGGGTTTTGAGTCACGCAGACCTGGACACAGCGAGAAACGGCAGTCCGGGCCCGTCCTCCCGCCCCGGGTTCTGCACCCCAGAGTTGGACGCTTGCTTCTCGCTTTCTCCTGAGAGAAGTCTCCGCGTTTCTGAACAATGCACTGCTAGTTTCTCATTTCTTTCTTTCTTTCTTTTTTTTTTTTTGGAGACGGAGCCTCGCCCTGTCGCCCAGGCTGGAGTGCAATGGCGCGATCTCGGCTCATTGCAACCTCCGCCTCCAGGGCTTAAACTATTCTCTTGCTTCAGCCTCCCGAGCAGC,334,111,450,MATPPPRGWAGQAPEVRHAGASASPTLRGAARFRAA,MATPPPRGWAGQAPEVRHAGASASPTLRGAARFRAA,36,2,True,ATG
26,c1riboseqorf720,CEP104,uORF,c1riboseqorf720:CEP104:uORF,c1riboseqorf720,GENCODE_Phase2,GENCODE_Phase2_Primary,1,ENST00000674879,ENST00000674879.1,ENSG00000116198,ENSG00000116198.14,CEP104-213,retained_intron,7030,46,220,AGGGCTGGGCTTGGTTGTCATGAGAGTGGAGGCTGCGGCCGGGGCT,ATGGGCAGAGGTGGCCGGGTCTGCTTCTTCCCTGTCACTGCCGCAGCAGCTTGGGGTGCAAGGCGGCCTGGGGCGGCGGAAGGGGCGCCCCCCGAGTTCCGTCTGAGGAAGCGGCGGCTGCCGAGGAAGTCCACGAGAGCCGGGTCCCGCACGCCCCTGGGGTCCGGGTGCTGA,GGCGCCGCCTGGCCGCCTCCCCGGGCCGCGGCACCGCTCCGACCCAGCCGCCCTCTCCACCGGGGCCGCGGCAGCGCCAGTCGGGGCCCTTCCCTCTAAGCAGCCCAAACGTGCAGAATGCCCCACAAGATTGGATTTGTAGTCGTCAGCTCATCTGGACACGAAGACGGCTTCAGTGCCCGGGAGCTCATGATCCACGCGCCAACTGTCAGTGGGTGGCGGTCACCTAGATTTTGCCAGTTTCCACAAGAAATTGTCCTTCAAATGGTGGAGAGATGTCGAATAAGGAAACTGCAGTTACTTGCTCACCAGTATATGATTTCAAGTAAAATTGAGTTCTACATTAGTGAAAGCTTGCCTGAATATTTTGCACCCTATCAAGCAGAGCGGTTTCGAAGACTTGGCTACGTGTCTCTCTGTGATAATGAAAAGACAGGTTGCAAAGCCCGG,46,174,450,MGRGGRVCFFPVTAAAAWGARRPGAAEGAPPEFRLRKRRLPRKSTRAGSRTPLGSGC,MGRGGRVCFFPVTAAAAWGARRPGAAEGAPPEFRLRKRRLPRKSTRAGSRTPLGSGC,57,26,True,ATG
27,c1ribos

(6178, 11)


,sample_id,protein_accession,transcript_accession,protein_len_aa,orf_seq,upstream_seq_50nt,downstream_seq_50nt,orf_nt_len,label_raw,protein_len_a,label_id
1,c1riboseqorf663,LINC01128,ENST00000670780,2,ATGGCATGA,GGGCGGGGTCTCGGGCAGGGGCGGGGAGCTTACCGACCTCCCGCCCCCGCTGCGCGCGTTTCTGGCCCTGCCAGTGTCTCCGCCGGTTGAAAGCGCGTGTCTGCGTCGGGTTCTGTTGGAGTGCGTTCGGTGCGCCGTGGGTCCGCGCTGCTTCCACCCAACTTCCTGTTAGTTTCTCTCCTGTCACA,TCTCGGCTCACTGCGACCTCTGCTTCCTGGGTTCAAGGATTCTCCTGCCTCAGCCTCCCAAGTAGCTGGGATTACAGGAATCTCACCAGTGACACGCGTTTGTGAAAAGCTGTGAATATTCAGTGCCACAGGGTTAAACCTCGGTACAGAAGCGAGACCCTGCTGGGCAGAGAAAGGGTGACCTGGAAGGTGTGTGAGGGCCCCGCCCATGGGCCCCTGTGATGGTCATGGCCTCAAGAATCACAGTCCAGGTGAGGCCCTGAATATCCACAGCCAGGACCAGTTCCCTCTGCTCTTGGGTGCCAAGGCCACCGTCAGCCAGGAGAGGCCTGCAGAATCAGCCTGTGCACCTCTCTGACGATAAGCAGAGATGCTGGTTTAGAGACTTGCTACTCACGGGGAAGTCCTTGGGCCTCCTGTACTCACATTACCTGGAACTTCTTGGAAATT,9,gencode,2,8
6,c1riboseqorf678,SDF4,ENST00000465727,4,ATGGCAGTGGCCTGA,GGCGTCCCCTCCGTGAGGTCGCGCCCGTTCGCACCGCCCCCGCCCGCAAGAAAG,TCCGGGCCCGTTGGCGGCGTCACTGACGCTTCGCTCCGGTCCTCGGATCCCGAGCGCGGGGAGGCAGACCGACTGTGAGCTGCTTGTCCCCATCCTGCGGCCGTCCTGGGGACACAGAGCCCTCCGTGGTGCCCGGGGATTGGATTGGAGCCAGGACCTCACTTCCTCCTCTGCCCCTGCCCCTGCCCCTCCCAGCACCTGGCCCACACCCTGCAGCCCGCCCCATGGTCTGGCCCTGGGTGGCGATGGCGTCCAGGTGGGGTCCCCTCATTGGCCTGGCTCCGTGCTGCCTCTGGCTCCTGGGGGCAGTCCTTCTGATGGACGCGTCTGCACGGCCTGCCAACCACTCGTCCACTCGAGAGAGAGTAGCCAACAGGGAGGAGAATGAGATCCTGCCCCCAGACCACCTGAACGGGGTGAAGCTGGAGATGGACGGGCACCTCAATCGCG,15,gencode,4,8
7,c1riboseqorf6,UBE2J2,ENST00000450390,32,ATGTGTGACTGGGCTTTGGAGGAGACTGGAGCCTCAGCCCTGTCCTGGCCACGGGCCGCTGGGGCTGGTGTGGGTGGGCCTTGTGTGCTGGATTTGTAG,TCCTGAAGTCGTGGAGGAGATTAAACAAAAACAGAAAGCACAAGACGAACTCAGTAGCAGACCCCAGACTCTCCCCTTGCCAGACGTGGTTCCAGACGGGGAGACGCACCTCGTCCAGAACGGGATTCAGCTGCTCAACGGGCATGCGCCGGGGGCCGTCCCAAACCTCGCAGGGCTCCAGCAGGCCAACCGGCACCACGGACTCCTGGGTGGCGCCCTGGCGAACTTGTTTGTGATAGTTGGGTTTGCAGCCTTTGCTTACACGGTCAAGTACGTGCTGAGGAGCATCGCGCAGGAGTGAGGCCCAGGCGCCGAGACCCAAGGCGCCACTGAGGGCACCGCGCACCAGAGCGTGACCTCGGCAGGCTGGACACACTGCCCAGCACAGGCAGACCCACCAGGCTCCTAGGTTTAGCTTTTAAAAACCTGAAAGGGGAAGCAAAAACCAAA,CTTATCTTCCGTGTTGTCTTTGGACCTGTTTTAGTAAACCCGTTTTTCATTTTATTAGATGTGGTCACTTAGAAATGCAAACTTGCTGCCGACCGCGGGCTGCTCCTGCGTTCTTGGAGCTCCTGGCGCGTTTCTCGGAGCTCCCGGCTCCTCAGCGGGTGGGAACCTCGGGGCCCAGGGGTGGAGCTGGCGTCCGCGGGTGCTGGTCTGGCCTGGCCGTGTGGTGATGAGGCTTAGCGGGGCCAGTGACGGCCGTGGCTCAGGATCCATAAGTCGGGGTTTGGTCTCAGCATTTACAAATGTGTTTACAGTCAGAATGAAACACATTCCTTCTAGAAAGTGCTTGGGGGTTTTTGCTGCCCTGGAAGCCAGGAGCCTGCTCACTCCAACCACAAGTCGCCCTTGACTGCGGCGGCCGCGAGCGGGGCGGGGGCTGCCGGTGCCCTCCGC,99,gencode,32,8
10,c1riboseqorf688,MRPL20,ENST00000482352,9,ATGGCAGCGGCGCTGCGCGTGCGTTGTTGA,TCCGACCCGGGCAAG,GTGTTCGGGACGCCGGCCTGCAGGCGCCATGGTCTTCCTCACCGCGCAGCTCTGGCTGCGGAATCGCGTCACCGACCGCTACTTTCGGATCCAGGAGGTGCTGAAGCACGCCAGGCACTTCCGGGGAAGGAAAAATCGCTGCTACAGGTTGGCGGTCAGAACCGTGATTCGAGCCTTTGTGAAATGCACCAAAGCCCGATACCTGAAGAAAAAGAACATGAGGACCCTCTGGATTAATCGAATTACAGCTGCTAGCCAGGAACATGGACTGAAGTATCCAGCGCTCATTGGGAATTTAGTTAAGGTATGGGTGAGTATGTGGGTCCCACTGAAATTCTGGACATCTGCAGAAACCATCATGCGGGGCGTGGTGGTGCTTCCTGTAGTCCCAGCTAATCAGGAGGCTGAGGCCAGAGGATCCCTGGAGACTGACTTCTGGGCTGTAGTTTG,30,gencode,9,8
11,c1riboseqorf312,ANKRD65,ENST00000454272,21,ATGGGGGGCTCCAGTTGGCCTGGGACACGGCTGGGGGCAGGCCAGAGGCCTCCACAGTGGATGTAG,A,GATCCCTGCTCCCCCCCAGATGGACTCCCAGAGGCCTGAGCCCAGAGAGGAGGAGGAGGAGGAACAGGAACTGCGGTGGATGGAGCTGGACTCCGAAGAGGCCCTGGGAACCAGGACAGAGGGGCCTAGTGTTGTCCAGGGCTGGGGGCACCTGCTCCAGGCCGTGTGGAGGGGCCCTGCAGGCCTGGTGACGCAGCTGCTGCGGCAAGGTGCCAGCGTGGAGGAGAGGGACCACGCAGGCCGGACCCCGCTCCACCTGGCCGTGCTGCGGGGCCACGCGCCCCTGGTGCGTCTCCTGCTGCAGCGAGGGGCCCCGGTGGGCGCGGTGGACCGGGCGGGGCGCACCGCGCTGCACGAGGCCGCCTGGCACGGACACTCGCGGGTGGCCGAGCTGCTGCTGCAGCGCGGGGCCTCGGCGGCGGCTCGCTCCGGGACGGGCCTCACGCCGCT,66,gencode,21,8


/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.pkl
/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.csv.gz


In [11]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_rnafm_token_nt450_ribotricer_openprot_transcript_5to3_v2.py \
  --input_table /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.pkl \
  --out_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_test_gencode \
  --local_model_dir /disk3/hejp/00_thesis/4_predict_translation/model/rnafm \
  --model_name rna-fm \
  --run_region all \
  --upstream_col upstream_seq_50nt \
  --orf_col orf_seq \
  --downstream_col downstream_seq_50nt \
  --id_col sample_id \
  --label_col label_raw \
  --min_aa 0 \
  --max_aa 150 \
  --dna_source coding \
  --batch_size 1 \
  --too_long skip \
  --device cuda

[INFO] RNA-FM token-level feature extraction
[INFO] input_table: /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.pkl
[INFO] out_dir: /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_test_gencode
[INFO] local_model_dir: /disk3/hejp/00_thesis/4_predict_translation/model/rnafm
[INFO] model_name: rna-fm
[INFO] run_region: all
[INFO] dna_source: coding
[INFO] device: cuda

[INFO] Loading input table...
[INFO] Input shape: (6178, 11)
[INFO] Columns: ['sample_id', 'protein_accession', 'transcript_accession', 'protein_len_aa', 'orf_seq', 'upstream_seq_50nt', 'downstream_seq_50nt', 'orf_nt_len', 'label_raw', 'protein_len_a', 'label_id']
[INFO] Infer aa length from ORF sequence

[INFO] Keep microproteins: 0 <= aa_len <= 150
[INFO] Remaining samples: 6178
[INFO] Filtered rows saved: /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_test_gencode/rnafm_filtered_0_150aa_rows.csv


In [12]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_esm2_token_embedding_nt450_ribotricer_openprot.py \
  --DATA_PATH /disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.pkl \
  --SAVE_DIR /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_esm2_token_test_gencode

Loading input table...
Input shape: (6178, 11)
Valid protein count: 6178
Loading ESM2 tokenizer/model...
Loading weights: 100%|███████████████████████| 566/566 [00:01<00:00, 431.72it/s]
EsmModel LOAD REPORT from: /disk3/hejp/00_thesis/4_predict_translation/model/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Model loaded on cuda:0
Proces

In [12]:
###加点阴性数据集进去###
import os, glob
import numpy as np
import pandas as pd

metadata_csv = "/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/train_pos_neg_merged.csv.gz"
rnafm_token_dir = "/disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_pos_neg"
run_dir = "/disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/rnafm_only"

out_dir = "/disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2"
os.makedirs(out_dir, exist_ok=True)

out_unused = f"{out_dir}/ribotricer_clean_unused_after_10runs.csv.gz"
out_10000 = f"{out_dir}/ribotricer_clean_unused_sample10000.csv.gz"

def clean_seq(x):
    return "" if pd.isna(x) else str(x).upper().replace("U", "T")

def add_bins(df):
    orf_bins = np.arange(90, 481, 30)
    flank_bins = np.arange(0, 501, 50)
    df["orf_bin"] = pd.cut(df["orf_len"], orf_bins, include_lowest=True)
    df["up_bin"] = pd.cut(df["up_len"], flank_bins, include_lowest=True)
    df["down_bin"] = pd.cut(df["down_len"], flank_bins, include_lowest=True)
    df = df.dropna(subset=["orf_bin", "up_bin", "down_bin"]).copy()
    df["len_bin"] = df["orf_bin"].astype(str) + "|" + df["up_bin"].astype(str) + "|" + df["down_bin"].astype(str)
    return df

# 1. 收集10个run里已经用过的ribotricer sample_id
used_ids = set()
for f in sorted(glob.glob(f"{run_dir}/translation_strict_lenmatched_run*.csv.gz")):
    x = pd.read_csv(f, usecols=lambda c: c in ["sample_id", "label_raw", "binary_label"])
    if "label_raw" in x.columns:
        x = x[x["label_raw"].astype(str).eq("ribotricer")]
    else:
        x = x[x["binary_label"].eq(0)]
    used_ids.update(x["sample_id"].astype(str))

print("10个run已用ribotricer:", len(used_ids))

# 2. 从总表读取并清洗ribotricer阴性
df = pd.read_csv(metadata_csv, low_memory=False)
df = df.loc[:, ~df.columns.duplicated()].copy()

df["sample_id"] = df["sample_id"].astype(str)
df["label_raw"] = df["label_raw"].astype(str)

df = df[df["label_raw"].eq("ribotricer")].copy()

df["orf_seq"] = df["orf_seq"].apply(clean_seq)
df["upstream_seq_50nt"] = df["upstream_seq_50nt"].apply(clean_seq).str[-450:]
df["downstream_seq_50nt"] = df["downstream_seq_50nt"].apply(clean_seq).str[:450]

df["orf_len"] = df["orf_seq"].str.len()
df["up_len"] = df["upstream_seq_50nt"].str.len()
df["down_len"] = df["downstream_seq_50nt"].str.len()

df = df[df["orf_len"].between(93, 453)].copy()
df = df[df["orf_seq"].str[-3:].isin(["TAA", "TAG", "TGA"])].copy()
df = add_bins(df)

df["binary_label"] = 0
df["label_id"] = 7

# 3. 加RNA-FM路径并过滤不存在的token
df["rnafm_up_path"] = df["sample_id"].apply(lambda x: f"{rnafm_token_dir}/upstream/{x}_upstream.pt")
df["rnafm_orf_path"] = df["sample_id"].apply(lambda x: f"{rnafm_token_dir}/orf/{x}_orf.pt")
df["rnafm_down_path"] = df["sample_id"].apply(lambda x: f"{rnafm_token_dir}/downstream/{x}_downstream.pt")

ok = (
    df["rnafm_up_path"].apply(os.path.exists) &
    df["rnafm_orf_path"].apply(os.path.exists) &
    df["rnafm_down_path"].apply(os.path.exists)
)
df = df[ok].copy()

# 4. 去掉10个run已经用过的样本，输出剩余总表
unused = df[~df["sample_id"].isin(used_ids)].copy()
unused = unused.drop_duplicates("sample_id").reset_index(drop=True)

unused.to_csv(out_unused, index=False, compression="gzip")
print("剩余未使用ribotricer:", unused.shape)
print("保存剩余总表:", out_unused)

# 5. 从剩余总表随机取10000个
if len(unused) < 10000:
    raise ValueError(f"剩余样本不足10000，目前只有 {len(unused)} 个")

sample10000 = unused.sample(n=10000, random_state=20260629).reset_index(drop=True)
sample10000.to_csv(out_10000, index=False, compression="gzip")

print("保存随机10000:", out_10000)
print(sample10000.shape)
display(sample10000.head())

10个run已用ribotricer: 41623
剩余未使用ribotricer: (34198, 22)
保存剩余总表: /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/ribotricer_clean_unused_after_10runs.csv.gz
保存随机10000: /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/ribotricer_clean_unused_sample10000.csv.gz
(10000, 22)


,sample_id,protein_accession,transcript_accession,protein_len_aa,orf_seq,upstream_seq_50nt,downstream_seq_50nt,orf_nt_len,label_raw,protein_len_a,...,up_len,down_len,orf_bin,up_bin,down_bin,len_bin,binary_label,rnafm_up_path,rnafm_orf_path,rnafm_down_path
0,ENST00000683811_21817807_21817920_114,ENST00000683811_21817807_21817920_114,ENST00000683811,38,ATGTATCCATTTTTTTTAGAAGAAATAAAGGGAAAAAAATCTCATT...,CCATTCGGGCCTTTAGGCATGAAACCAGATTTAAACAACGTATGCT...,ATTCATAACAAGATTACATCTTGCAAATAATAGTAGATTGGGGCCC...,117,ribotricer,38,...,450,450,"(89.999, 120.0]","(400.0, 450.0]","(400.0, 450.0]","(89.999, 120.0]|(400.0, 450.0]|(400.0, 450.0]",0,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...
1,ENST00000539618_132680598_132681146_99,ENST00000539618_132680598_132681146_99,ENST00000539618,33,ATGCATCCTACCGAGATTTTAGATGAAGATAAGCGCTTAGGCAGTG...,GGATGATGGCGCCACTTCCTCAGTTTCGGCACTCAAGCGCCTGGAA...,CTGCGACAAGCTAAAACCCACTTTTATGGGTGTTCATGTAGCTCTG...,102,ribotricer,33,...,133,64,"(89.999, 120.0]","(100.0, 150.0]","(50.0, 100.0]","(89.999, 120.0]|(100.0, 150.0]|(50.0, 100.0]",0,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...
2,ENST00000642619_29192899_29193030_132,ENST00000642619_29192899_29193030_132,ENST00000642619,44,ATGATGCAGTTCCTAATCTGGAGCCTTCAGGGACACGTGACCCATC...,GCAAAAATGGAAAGTGTTCACATTGGGGTAGTTCAACTTTTTAAAA...,TAATCTATGTAGCCCCCCAGATCTTTAGATGGTGGACTATCTGACT...,135,ribotricer,44,...,450,450,"(120.0, 150.0]","(400.0, 450.0]","(400.0, 450.0]","(120.0, 150.0]|(400.0, 450.0]|(400.0, 450.0]",0,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...
3,ENST00000662319_20931310_20931417_108,ENST00000662319_20931310_20931417_108,ENST00000662319,36,ATGCTAGTAAGAGCAGAAACTGTGTTGGGAGGAGGGAAGATGTATG...,GACAATGACATCATTCAAATACTCTAAAAAGCTCATATTATAGAAT...,ATTTTTCAATTATTCCAGTTATTCTCTATAAATTAGCATACCTATT...,111,ribotricer,36,...,450,450,"(89.999, 120.0]","(400.0, 450.0]","(400.0, 450.0]","(89.999, 120.0]|(400.0, 450.0]|(400.0, 450.0]",0,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...
4,ENST00000315994_103601869_103601994_126,ENST00000315994_103601869_103601994_126,ENST00000315994,42,ATGTGTACGTCTCTATCGCAGACTACGAGGGGGATGAGGAGACAGC...,GCCTGGAGAAGATCGAGAGGCGCGTCCAAGCACTGAACACCGTCAA...,AGCCCTTCAAAGGCTGGGTGCCTTCCAACTACCTTGAGAAAAAGAA...,129,ribotricer,42,...,450,450,"(120.0, 150.0]","(400.0, 450.0]","(400.0, 450.0]","(120.0, 150.0]|(400.0, 450.0]|(400.0, 450.0]",0,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...,/disk4hu/hejp/00_thesis/4_predict_translation/...


In [13]:
import os
import pandas as pd

df_gencode_neg1w=pd.read_csv('/disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/ribotricer_clean_unused_sample10000.csv.gz')
df_gencode_pos=pd.read_csv('/disk3/hejp/00_thesis/4_predict_translation/data/processed/dataset_nt450_v3/test_table_gencode.csv.gz')
print(df_gencode_pos['label_id'].value_counts())
print(df_gencode_neg1w['label_id'].value_counts())
print(min(df_gencode_neg1w['protein_len_aa']),max(df_gencode_neg1w['protein_len_aa']))
print(min(df_gencode_pos['protein_len_aa']),max(df_gencode_pos['protein_len_aa']))

cols = [
    "sample_id", "protein_accession", "transcript_accession",
    "protein_len_aa", "orf_seq", "upstream_seq_50nt", "downstream_seq_50nt",
    "orf_nt_len", "label_raw", "protein_len_a", "label_id"
]

out_csv = "/disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/gencode_pos_ribotricer_neg1w_merged.csv.gz"

# 去重复列
df_gencode_pos = df_gencode_pos.loc[:, ~df_gencode_pos.columns.duplicated()].copy()
df_gencode_neg1w = df_gencode_neg1w.loc[:, ~df_gencode_neg1w.columns.duplicated()].copy()

# 统一 protein_len_aa / protein_len_a
for df in [df_gencode_pos, df_gencode_neg1w]:
    if "protein_len_aa" not in df.columns and "protein_len_a" in df.columns:
        df["protein_len_aa"] = df["protein_len_a"]
    if "protein_len_a" not in df.columns and "protein_len_aa" in df.columns:
        df["protein_len_a"] = df["protein_len_aa"]
    if "orf_nt_len" not in df.columns:
        df["orf_nt_len"] = df["orf_seq"].astype(str).str.len()
        
df_merge = pd.concat(
    [
        df_gencode_pos.reindex(columns=cols),
        df_gencode_neg1w.reindex(columns=cols)
    ],
    axis=0,
    ignore_index=True
)
# 去重并保存
df_merge = df_merge.drop_duplicates("sample_id").reset_index(drop=True)

os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_merge.to_csv(out_csv, index=False, compression="gzip")

print("保存成功:", out_csv)
print("合并后 shape:", df_merge.shape)
print(df_merge["label_raw"].value_counts())
display(df_merge.head())

label_id
8    6178
Name: count, dtype: int64
label_id
7    10000
Name: count, dtype: int64
30 150
1 144
保存成功: /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/gencode_pos_ribotricer_neg1w_merged.csv.gz
合并后 shape: (16178, 11)
label_raw
ribotricer    10000
gencode        6178
Name: count, dtype: int64


,sample_id,protein_accession,transcript_accession,protein_len_aa,orf_seq,upstream_seq_50nt,downstream_seq_50nt,orf_nt_len,label_raw,protein_len_a,label_id
0,c1riboseqorf663,LINC01128,ENST00000670780,2,ATGGCATGA,GGGCGGGGTCTCGGGCAGGGGCGGGGAGCTTACCGACCTCCCGCCC...,TCTCGGCTCACTGCGACCTCTGCTTCCTGGGTTCAAGGATTCTCCT...,9,gencode,2,8
1,c1riboseqorf678,SDF4,ENST00000465727,4,ATGGCAGTGGCCTGA,GGCGTCCCCTCCGTGAGGTCGCGCCCGTTCGCACCGCCCCCGCCCG...,TCCGGGCCCGTTGGCGGCGTCACTGACGCTTCGCTCCGGTCCTCGG...,15,gencode,4,8
2,c1riboseqorf6,UBE2J2,ENST00000450390,32,ATGTGTGACTGGGCTTTGGAGGAGACTGGAGCCTCAGCCCTGTCCT...,TCCTGAAGTCGTGGAGGAGATTAAACAAAAACAGAAAGCACAAGAC...,CTTATCTTCCGTGTTGTCTTTGGACCTGTTTTAGTAAACCCGTTTT...,99,gencode,32,8
3,c1riboseqorf688,MRPL20,ENST00000482352,9,ATGGCAGCGGCGCTGCGCGTGCGTTGTTGA,TCCGACCCGGGCAAG,GTGTTCGGGACGCCGGCCTGCAGGCGCCATGGTCTTCCTCACCGCG...,30,gencode,9,8
4,c1riboseqorf312,ANKRD65,ENST00000454272,21,ATGGGGGGCTCCAGTTGGCCTGGGACACGGCTGGGGGCAGGCCAGA...,A,GATCCCTGCTCCCCCCCAGATGGACTCCCAGAGGCCTGAGCCCAGA...,66,gencode,21,8


In [14]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_rnafm_token_nt450_ribotricer_openprot_transcript_5to3_v2.py \
  --input_table /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/gencode_pos_ribotricer_neg1w_merged.csv.gz \
  --out_dir /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_test_gencode_ribo1w \
  --local_model_dir /disk3/hejp/00_thesis/4_predict_translation/model/rnafm \
  --model_name rna-fm \
  --run_region all \
  --upstream_col upstream_seq_50nt \
  --orf_col orf_seq \
  --downstream_col downstream_seq_50nt \
  --id_col sample_id \
  --label_col label_raw \
  --min_aa 0 \
  --max_aa 500 \
  --dna_source coding \
  --batch_size 1 \
  --too_long skip \
  --device cuda

[INFO] RNA-FM token-level feature extraction
[INFO] input_table: /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/gencode_pos_ribotricer_neg1w_merged.csv.gz
[INFO] out_dir: /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_rnafm_token_test_gencode_ribo1w
[INFO] local_model_dir: /disk3/hejp/00_thesis/4_predict_translation/model/rnafm
[INFO] model_name: rna-fm
[INFO] run_region: all
[INFO] dna_source: coding
[INFO] device: cuda

[INFO] Loading input table...
[INFO] Input shape: (16178, 11)
[INFO] Columns: ['sample_id', 'protein_accession', 'transcript_accession', 'protein_len_aa', 'orf_seq', 'upstream_seq_50nt', 'downstream_seq_50nt', 'orf_nt_len', 'label_raw', 'protein_len_a', 'label_id']
[INFO] Infer aa length from ORF sequence

[INFO] Keep microproteins: 0 <= aa_len <= 500
[INFO] Remaining samples: 16178
[INFO] Filtered rows saved: /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_

In [17]:
!python /disk3/hejp/00_thesis/4_predict_translation/src/build_features_esm2_token_embedding_nt450_ribotricer_openprot.py \
  --DATA_PATH /disk4hu/hejp/00_thesis/4_predict_translation/results/training_result/training_v3_2/gencode_independent_v2/gencode_pos_ribotricer_neg1w_merged.csv.gz \
  --SAVE_DIR /disk4hu/hejp/00_thesis/4_predict_translation/results/features/features_v3_esm2_token_test_gencode_ribo1w

Loading input table...
Input shape: (16178, 11)
Valid protein count: 16178
Loading ESM2 tokenizer/model...
Loading weights: 100%|██████████████████████| 566/566 [00:00<00:00, 7349.16it/s]
EsmModel LOAD REPORT from: /disk3/hejp/00_thesis/4_predict_translation/model/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Model loaded on cuda:0
Proc

In [ ]:
###看原始的primary.fna文件的orf的长度分布###
from Bio import SeqIO
import pandas as pd
import gzip

orf_fna = "/disk3/hejp/00_thesis/4_predict_translation/dataset/gencode/phase2primary/Ribo-seq_ORFs.primary.fna"

def open_maybe_gz(path):
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path, "r")

def clean_seq(seq):
    return str(seq).upper().replace("U", "T")

def get_aa_len_from_fna(seq):
    seq = clean_seq(seq)
    nt_len = len(seq)
    stop_codons = {"TAA", "TAG", "TGA"}
    
    if nt_len >= 3 and seq[-3:] in stop_codons:
        return (nt_len - 3) // 3
    else:
        return nt_len // 3

rows = []

with open_maybe_gz(orf_fna) as f:
    for rec in SeqIO.parse(f, "fasta"):
        seq = clean_seq(rec.seq)
        aa_len = get_aa_len_from_fna(seq)
        
        rows.append({
            "orf_id": rec.id.split()[0],
            "nt_len": len(seq),
            "aa_len": aa_len,
            "start_codon": seq[:3],
            "stop_codon": seq[-3:],
            "keep_30_150aa": 30 <= aa_len <= 150,
        })

df = pd.DataFrame(rows)

print("总ORF数量:", len(df))
print("aa长度在30-150之间的ORF数量:", df["keep_30_150aa"].sum())
print("比例:", df["keep_30_150aa"].mean())

print("\naa_len描述:")
print(df["aa_len"].describe())

print("\n起始密码子统计:")
print(df["start_codon"].value_counts().head(20))

print("\n终止密码子统计:")
print(df["stop_codon"].value_counts().head(20))

总ORF数量: 10127
aa长度在30-150之间的ORF数量: 1847
比例: 0.1823837266712748

aa_len描述:
count    10127.000000
mean        18.407623
std         18.797599
min          1.000000
25%          6.000000
50%         12.000000
75%         24.000000
max        219.000000
Name: aa_len, dtype: float64

起始密码子统计:
start_codon
ATG    6184
CTG    1351
GTG     716
TTG     414
AGG     367
AAG     334
ACG     203
ATC     192
ATT     190
ATA     146
GCG       3
TGG       3
GGG       2
GCT       2
CTT       2
GAG       2
TCC       2
CTC       2
CGA       1
TCA       1
Name: count, dtype: int64

终止密码子统计:
stop_codon
TGA    5315
TAG    2535
TAA    2277
Name: count, dtype: int64


In [ ]:

table_gencode_phase2_slim